# Model Architecture before and after Quantization

In [1]:
# !pip install peft

In [5]:
# Imports

import os
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datetime import datetime
from peft import LoraConfig, PeftModel

In [4]:
# Constants

BASE_MODEL = 'meta-llama/Llama-3.2-3B'

# Below fine-tuned model is provided by course instructor, Ed Donner, Imported here for Architecture and LoRA Parameter Overview:
FINETUNED_MODEL = f"ed-donner/price-2025-11-30_15.10.55-lite"

In [6]:
# Login to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Trying out Different Quantization on Base Model:

### 1. Base Model without Quantization:

In [12]:
base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    device_map= 'auto')

print(f"Memory FootPrint of '{BASE_MODEL}' is {base_model.get_memory_footprint() / 1e9:,.2f}GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory FootPrint of 'meta-llama/Llama-3.2-3B' is 6.43GB


In [13]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (

### 2. Base Model with 8bit Quantization:

In [14]:
quant_config = BitsAndBytesConfig(
    load_in_8bit= True
)

base_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path= BASE_MODEL,
    quantization_config= quant_config,
    device_map= 'auto'
)

print(f"Memory FootPrint of '{BASE_MODEL}' is {base_model.get_memory_footprint() / 1e9:,.2f}GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory FootPrint of 'meta-llama/Llama-3.2-3B' is 3.61GB


In [ ]:
base_model

### Base Model with 4bit Quantization:

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_use_double_quant= True,
    bnb_4bit_compute_dtype= torch.bfloat16,
    bnb_4bit_quant_type= 'nf4'
)

base_model= AutoModelForCausalLM.from_pretrained(
    model_name_or_path= BASE_MODEL,
    quantization_config= quant_config,
    device_map= 'auto'
)

print(f"Memory FootPrint of '{BASE_MODEL}' is {base_model.get_memory_footprint() / 1e9}GB")

In [ ]:
base_model

## Fine-Tuned Model:

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(model= base_model, model_id= FINETUNED_MODEL)

print(f"Memory FootPrint of '{FINETUNED_MODEL}' is {fine_tuned_model.get_memory_footprint() / 1e9}GB")